# 🏥 Insurance Policy Wording Extractor
**PyMuPDF · Modular Pipeline · RAG-Ready**

This notebook walks through:
1. Installation & setup
2. Upload your own PDF (or use the sample HDFC Ergo policy)
3. Run the extraction pipeline
4. Inspect chunk quality & statistics
5. RAG query demo (no API key needed — pure retrieval)

---

## 0 · Setup

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────
!pip install pymupdf4llm chromadb sentence-transformers rank_bm25 --quiet
!pip install langchain-core langchain-text-splitters langchain-chroma langchain-huggingface langchain-community langchain-google-genai --quiet 
print('✅ Dependencies installed')

In [ ]:
import sys, os, json, textwrap, time
from pathlib import Path
import importlib

# ── If cloned from GitHub ─────────────────────────────────────────────────
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

# ── Clone and switch context ──────────────────────────────────────────────
if not os.path.exists('Insurance-RAG'):
    !git clone https://{token}@github.com/falcon978/Insurance-RAG
    %cd Insurance-RAG
else:
    %cd Insurance-RAG

# ── Load new workflow components ──────────────────────────────────────────
from rag_ingestion.pipeline import ExtractionPipeline
from rag_ingestion.extractor import PDFExtractor
from rag_ingestion.cleaner import clean_markdown_layout
from rag_ingestion.chunker import MarkdownHierarchicalChunker

print('✅ Packages and layout-aware components imported successfully')

---
## 1 · Load a PDF
Two options — run whichever applies to you.

In [ ]:
# ── Option A: Upload your own PDF ─────────────────────────────────────────
from google.colab import files
uploaded = files.upload()           # opens file picker
PDF_PATH = list(uploaded.keys())[0]
print(f'Using: {PDF_PATH}')

In [ ]:
# ── Option B: Download the sample HDFC Ergo Optima Secure policy ──────────
import urllib.request

SAMPLE_URL = (
    'https://customer-portal-assets.hdfcergo.com/assets/v2/docs/'
    'default-source/downloads/policy-wordings/health/'
    'optima-secure-revision/optima-secure-revision-pw-647504209314.pdf'
)
PDF_PATH = 'optima-secure.pdf'

print('Downloading …')
urllib.request.urlretrieve(SAMPLE_URL, PDF_PATH)
print(f'✅ Saved to {PDF_PATH}  ({os.path.getsize(PDF_PATH) // 1024} KB)')

In [ ]:
# ── Option B: Download the sample Care Supreme policy ──────────
# import urllib.request

# SAMPLE_URL = (
#     'https://cms.careinsurance.com/cms/public/uploads/download_center/care-supreme---policy-terms-&-conditions-(effective-from-19-march-2025).pdf?rv=0.86869200%201775054695'
# )
PDF_PATH = 'care-supreme.pdf'

# print('Downloading …')
# urllib.request.urlretrieve(SAMPLE_URL, PDF_PATH)
# print(f'✅ Saved to {PDF_PATH}  ({os.path.getsize(PDF_PATH) // 1024} KB)')

---
## 2 · Extraction Pipeline Walkthrough
We run each phase separately so you can inspect what's happening at every step.

### 2a · Phase 1 — Raw block extraction

In [ ]:
import importlib

# Import the modules
from rag_ingestion import extractor

# Reload the modules to reflect any code changes
# importlib.reload(patterns)
# importlib.reload(models)
# importlib.reload(pipeline)
importlib.reload(extractor)
# importlib.reload(reconstructor)
# importlib.reload(chunker)
# importlib.reload(cleaner)
# Import the specific classes/functions after reload
# from rag_ingestion.pipeline import ExtractionPipeline
from rag_ingestion.extractor import PDFExtractor
# from rag_ingestion.reconstructor import SectionReconstructor
# from rag_ingestion.chunker import Chunker
# from rag_ingestion.cleaner import clean_block_text

print("Modules reloaded successfully!")

In [ ]:
# Initialize extractor and pull raw text layout
extractor = PDFExtractor(str(PDF_PATH))
doc_meta  = extractor.get_document_metadata()
toc       = extractor.get_toc()
md_text   = extractor.extract_markdown_with_pages()

print('Document metadata:')
for k, v in doc_meta.items():
    print(f'  {k:15}: {v}')
print(f'Total Raw Markdown Length: {len(md_text)} characters')

In [ ]:
from rag_ingestion.cleaner import clean_markdown_layout

# Run layout optimizations and regex header promotions
cleaned_md = clean_markdown_layout(md_text)

print(f"Raw Markdown character count     : {len(md_text)}")
print(f"Cleaned Markdown character count : {len(cleaned_md)}")

# Isolate lines matching our newly promoted H3 headers (###)
promoted_subclauses = [line for line in cleaned_md.split('\n') if line.startswith('###')]

print(f"\n✅ Total structural sub-clauses promoted to H3: {len(promoted_subclauses)}")
print('\nPreviewing the first 20 promoted sub-clauses and inline named headers:')
print('─' * 80)
for heading in promoted_subclauses[:20]:
    print(heading)

In [ ]:
# ── Inspect block type distribution ──────────────────────────────────────
from collections import Counter

type_counts = Counter(b.block_type for b in raw_blocks)
print('Block types:')
for btype, count in type_counts.most_common():
    bar = '█' * (count // 10)
    print(f'  {btype:10} {count:4}  {bar}')

In [ ]:
# ── Inspect a few raw blocks from a specific page ─────────────────────────
PAGE_TO_INSPECT = 4   # change this

page_blocks = [b for b in raw_blocks if b.page_num == PAGE_TO_INSPECT]
print(f'Blocks on page {PAGE_TO_INSPECT}:')
for b in page_blocks[:10]:
    preview = b.text[:80].replace('\n', ' ')
    print(f'  [{b.block_type:8}] font={b.font_size:.1f} bold={b.is_bold}  "{preview}"')

In [ ]:
# ── See what cleaning does to a single block ──────────────────────────────
sample_block = raw_blocks[5]   # pick any block index
from rag_ingestion.cleaner import clean_block_text
print('--- RAW ---')
print(repr(sample_block.text))
print('\n--- CLEANED ---')
print(repr(clean_block_text(sample_block.text)))

### 2b · Phase 2 — Section reconstruction

In [ ]:
reconstructor = SectionReconstructor()
sections      = reconstructor.reconstruct(raw_blocks)

print(f'Sections found: {len(sections)}\n')

print(f'{"#":>3}  {"Section (Parent)":<25} | {"Heading/Term":<30} | {"Pages":<12} | {"Chars"}')
print('-' * 85)
for i, s in enumerate(sections):
    # Use sub_section if available, otherwise a snippet of the heading
    sub = s["sub_section"]
    pages = f"{min(s['pages'])}–{max(s['pages'])}"
    print(f'{i:>3}  {s["section"][:25]:<25} | {sub:<30} | {pages:<12} | {len(s["text"])}')

In [ ]:
# ── Inspect a specific section ────────────────────────────────────────────
SECTION_IDX = 9   # change to any index from the table above

s = sections[SECTION_IDX]
print(f'Section  : {s["section"]}')
print(f'Heading  : {s["heading"]}')
print(f'Pages    : {sorted(s["pages"])}')
print(f'Chars    : {len(s["text"])}')
print()
print(s['text'][:1000])

### 2c · Phase 3 — Chunking

In [ ]:
import importlib

# Import the modules
from rag_ingestion import pipeline, extractor, reconstructor, chunker, cleaner, models

# Reload the modules to reflect any code changes
importlib.reload(models)
importlib.reload(pipeline)
importlib.reload(extractor)
importlib.reload(reconstructor)
importlib.reload(chunker)
importlib.reload(cleaner)
# Import the specific classes/functions after reload
from rag_ingestion.pipeline import ExtractionPipeline
from rag_ingestion.extractor import PDFExtractor
from rag_ingestion.reconstructor import SectionReconstructor
from rag_ingestion.chunker import HierarchicalChunker
from rag_ingestion.cleaner import clean_block_text

print("Modules reloaded successfully!")

In [ ]:
CHUNK_SIZE = 1200   # target chars per chunk — tune this
OVERLAP    = 180    # overlap chars between chunks

chunker = HierarchicalChunker(
    chunk_size  = CHUNK_SIZE,
    chunk_overlap = OVERLAP,
    source_file = PDF_PATH,
)
chunks = chunker.chunk(sections)
# chunks = chunker.merge_short_chunks(chunks, min_chars=CHUNK_SIZE//5)  # merge chunks shorter than min chars

# Fix: The 'chunks' variable is a list, which does not have a .save() method.
# We need to manually save the list of PolicyChunk objects to a JSON file.

# 'json' and 'Path' are already imported in an earlier cell (64FU7dNJirAr).

output_dir = Path('data/chunks')
output_dir.mkdir(parents=True, exist_ok=True)
out_path = output_dir / 'policy_chunks.json'

# Convert each PolicyChunk object to a dictionary for JSON serialization
chunks_as_dicts = [{
    "chunk_id": c.chunk_id,
    "source_file": c.source_file,
    "page_start": c.page_start,
    "page_end": c.page_end,
    "section": c.section,
    "heading": c.heading,
    "text": c.text,
    "token_estimate": c.token_estimate,
    "metadata": c.metadata
} for c in chunks]

with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(chunks_as_dicts, f, indent=2)

print(f'Saved → {out_path}')
print(f'Total chunks: {len(chunks)}')
print(f'Avg chars   : {sum(len(c.text) for c in chunks) // len(chunks)}')
print(f'Avg tokens  : ~{sum(c.token_estimate for c in chunks) // len(chunks)}')


In [ ]:
# Standard parameters for your tight, isolated chunk distribution
CHUNK_SIZE = 600   # Lower character limit to keep exclusions semantically isolated
OVERLAP    = 100

chunker = MarkdownHierarchicalChunker(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP,
)

# Build the document slices
chunks = chunker.chunk(cleaned_md, source_file=PDF_PATH)

# Package data structure to disk 
output_dir = Path('data/chunks')
output_dir.mkdir(parents=True, exist_ok=True)
out_path = output_dir / 'policy_chunks.json'

chunks_as_dicts = [{
    "chunk_id": c.chunk_id,
    "source_file": c.source_file,
    "page_start": c.page_start,
    "page_end": c.page_end,
    "section": c.section,
    "sub_section": c.sub_section,
    "heading": c.heading,
    "text": c.text,
    "token_estimate": c.token_estimate,
    "metadata": c.metadata
} for c in chunks]

with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(chunks_as_dicts, f, indent=2)

print(f'Saved chunks successfully → {out_path}')
print(f'Total chunks generated   : {len(chunks)}')
print(f'Avg chars per chunk      : {sum(len(c.text) for c in chunks) // len(chunks)}')
print(f'Avg token estimate       : ~{sum(c.token_estimate for c in chunks) // len(chunks)}')

---
## 3 · Chunk Quality Inspection

In [ ]:
!pip install matplotlib --quiet

In [ ]:
# ── Size distribution ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

char_counts = [len(c.text) for c in chunks]

plt.figure(figsize=(10, 4))
plt.hist(char_counts, bins=30, color='steelblue', edgecolor='white')
plt.axvline(CHUNK_SIZE, color='red',    linestyle='--', label=f'Target ({CHUNK_SIZE})')
plt.axvline(sum(char_counts)/len(char_counts), color='orange', linestyle='--', label='Mean')
plt.xlabel('Characters per chunk')
plt.ylabel('Count')
plt.title('Chunk Size Distribution')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Min: {min(char_counts)}  Max: {max(char_counts)}  Mean: {sum(char_counts)//len(char_counts)}')

In [ ]:
# ── Section coverage ──────────────────────────────────────────────────────
from collections import Counter

section_counts = Counter(c.section for c in chunks)
print(f'{"Section":<45}  {"Chunks"}')
print('-' * 55)
for section, count in section_counts.most_common(15):
    bar = '█' * count
    print(f'{section[:45]:<45}  {count:3}  {bar}')

In [ ]:
# ── Flag counts — useful for checking table / list detection ──────────────
has_table = sum(1 for c in chunks if c.metadata['has_table'])
has_list  = sum(1 for c in chunks if c.metadata['has_list'])
print(f'Chunks with detected tables  : {has_table}')
print(f'Chunks with detected lists   : {has_list}')

In [ ]:
# Change this index to scan through specific clause chunks
CHUNK_IDX = 15  

c = chunks[CHUNK_IDX]
print(f'Chunk ID    : {c.chunk_id}')
print(f'Section     : {c.section}')
print(f'Sub-Section : {c.sub_section}')
print(f'Pages       : {c.page_start} – {c.page_end}')
print(f'Tokens ~    : {c.token_estimate}')
print(f'Metadata    : {c.metadata}')
print('\n--- TEXT PAYLOAD WITH CONTEXT INJECTION ---')
print(c.text)

In [ ]:
# ── Flag very short chunks (possible extraction noise) ────────────────────
short_chunks = [c for c in chunks if len(c.text) < 100]
print(f'Chunks under 100 chars: {len(short_chunks)}')
for c in short_chunks[:5]:
    print(f'  [{c.section[:30]}]  "{c.text[:80]}"')

---
## 4 · Save Chunks to JSON

In [ ]:
# ── Run the full pipeline in one call (wraps all 3 phases) ────────────────
result = ExtractionPipeline(
    pdf_path   = PDF_PATH,
    chunk_size = CHUNK_SIZE,
    overlap    = OVERLAP,
).run()

out_path = result.save('data/chunks/policy_chunks.json')
print(f'Saved → {out_path}')

In [ ]:
# ── Download the JSON (Colab only) ────────────────────────────────────────
from google.colab import files
files.download('data/chunks/policy_chunks.json')

---
## 5 · RAG Query Demo
Pure retrieval — no LLM API key needed. Uses sentence-transformers + ChromaDB.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# ── Build ChromaDB collection ─────────────────────────────────────────────
client = chromadb.Client()   # in-memory for the demo

emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='all-MiniLM-L6-v2'
)

collection = client.get_or_create_collection(
    name='insurance_policy',
    embedding_function=emb_fn,
)

# Upsert in batches of 100
BATCH = 100
for i in range(0, len(chunks), BATCH):
    batch = chunks[i: i + BATCH]
    collection.upsert(
        ids       = [c.chunk_id for c in batch],
        documents = [c.text     for c in batch],
        metadatas = [{
            'section'   : c.section,
            'heading'   : c.heading[:100],
            'page_start': c.page_start,
            'page_end'  : c.page_end,
            'has_table' : str(c.metadata['has_table']),
        } for c in batch],
    )

print(f'✅ ChromaDB collection ready — {collection.count()} documents')

In [ ]:
def retrieve(query: str, n: int = 3, section_filter: str = None):
    """
    Retrieve top-n chunks for a query.
    Optionally restrict to chunks whose section contains section_filter.
    """
    where = None
    if section_filter:
        # ChromaDB where filter — exact match on the stored section string
        # For partial match, post-filter after retrieval (see cell below)
        pass

    results = collection.query(
        query_texts = [query],
        n_results   = n,
        include     = ['documents', 'metadatas', 'distances'],
    )

    docs      = results['documents'][0]
    metadatas = results['metadatas'][0]
    distances = results['distances'][0]

    # Post-filter by section keyword if provided
    if section_filter:
        filtered = [
            (d, m, dist) for d, m, dist in zip(docs, metadatas, distances)
            if section_filter.lower() in m['section'].lower()
        ]
        docs, metadatas, distances = zip(*filtered) if filtered else ([], [], [])

    return list(zip(docs, metadatas, distances))


def print_results(query, results):
    print(f'Query: "{query}"')
    print('─' * 70)
    for i, (doc, meta, dist) in enumerate(results):
        score = round(1 - dist, 3)
        print(f'\n[{i+1}] Section : {meta["section"]}')
        print(f'    Heading : {meta["heading"][:60]}')
        print(f'    Pages   : {meta["page_start"]}–{meta["page_end"]}')
        print(f'    Score   : {score}')
        print(f'    Text    : {doc[:300].replace(chr(10), " ")} …')

In [ ]:
# ── Try any insurance query ───────────────────────────────────────────────
QUERIES = [
    'What is the waiting period for pre-existing diseases?',
    'Is maternity benefit covered?',
    'What are the room rent sub-limits?',
    'How do I file a cashless claim?',
    'What conditions are excluded from coverage?',
]

for q in QUERIES:
    results = retrieve(q, n=2)
    print_results(q, results)
    print('\n' + '=' * 70 + '\n')

In [ ]:
# ── Section-filtered retrieval ────────────────────────────────────────────
# Only retrieve from chunks whose section contains 'EXCLUSION'

results = retrieve(
    'pre-existing disease',
    n=5,
    section_filter='EXCLUSION'
)
print_results('pre-existing disease  [filtered: EXCLUSION sections only]', results)

In [ ]:
# ── Your own query ────────────────────────────────────────────────────────
MY_QUERY = 'your question here'

results = retrieve(MY_QUERY, n=3)
print_results(MY_QUERY, results)